In [ ]:
from scipy.optimize import linprog

# 5 Вариант
#Сделал всё на русском, мой B1 уровень нее потянет эту схватку

# Коэффициенты целевой функции (инвертированы для максимизации
c = [-11, -14, -9, -13, -12, -17]

# Матрица коэффициентов левой части ограничений
A_ub = [
    [3, 4, 2, 5, 2, 6],       # Ограничение 1
    [2, 3, 1, 4, 1, 5],       # Ограничение 2
    [15, 20, 10, 25, 8, 30],  # Ограничение 3
    [-1, -1, -1, 0, 0, 0],    # Ограничение 4 (умножено на -1 для приведения к <=)
    [0, 0, 0, -1, -1, -1],    # Ограничение 5 (умножено на -1 для приведения к <=)
    [0, 0, 0, 1, 0, 1]        # Ограничение 6
]

# Вектор правых частей ограничений
b_ub = [210, 150, 800, -14, -11, 16]

# Границы переменных (x_j >= 0)
bounds = [(0, None) for _ in range(6)]

# Решение симплекс-методом
res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')

print("--- ЗАДАЧА 1: РЕЗУЛЬТАТЫ ---")
if res.success:
    print("Оптимальный вектор X*:")
    for i, val in enumerate(res.x):
        print(f"x{i+1} = {val:.3f}")
    print(f"Максимальное значение F(X*) = {-res.fun:.3f}")
    
    print("\nТеневые цены (двойственные оценки) ограничений:")
    for idx, shadow_p in enumerate(res.ineqlin.marginals):
        print(f"Ограничение {idx+1}: {abs(shadow_p):.3f}")

--- ЗАДАЧА 1: РЕЗУЛЬТАТЫ ---
Оптимальный вектор X*:
x1 = 0.000
x2 = 0.000
x3 = 14.000
x4 = 0.000
x5 = 82.500
x6 = 0.000
Максимальное значение F(X*) = 1116.000

Теневые цены (двойственные оценки) ограничений:
Ограничение 1: 0.000
Ограничение 2: 0.000
Ограничение 3: 1.500
Ограничение 4: 6.000
Ограничение 5: 0.000
Ограничение 6: 0.000


Максимальное значение функции: F(X) = 1116.00.
В ходе симплекс-оптимизации найден глобальный максимум целевой функции. Расчёт показал, что компоненты x_3 и x_5 принимают граничные значения, обусловленные структурными ограничениями 3 и 4 (x_3 = 14, x_5 = 82.5). Анализ двойственных оценок (теневых цен) выявил, что ограничивающими факторами системы выступают третье и четвёртое неравенства (оценки 1.5 и 6 соответственно); изменение их правых частей окажет наибольшее влияние на прирост целевой функции.

In [5]:
import numpy as np

# Начальные параметры модели
X_START = 10.0
N_STEPS = 5
ALPHA = 0.45
BETA = 0.55

# Формирование одномерной сетки состояний с шагом 1
x_grid = np.arange(0, 11, 1)

# Матрицы для хранения значений функции Беллмана и управлений
F_table = np.zeros((N_STEPS + 1, len(x_grid)))
Y_table = np.zeros((N_STEPS + 1, len(x_grid)))

# Функция линейной интерполяции для перехода между узлами сетки
def get_interpolated_cost(x_val, step_idx):
    if x_val <= 0: return F_table[step_idx, 0]
    if x_val >= 10: return F_table[step_idx, 10]
    
    idx_low = int(np.floor(x_val))
    idx_high = int(np.ceil(x_val))
    if idx_low == idx_high: return F_table[step_idx, idx_low]
    
    weight = (x_val - idx_low) / (idx_high - idx_low)
    return F_table[step_idx, idx_low] + weight * (F_table[step_idx, idx_high] - F_table[step_idx, idx_low])

# Прямой ход динамического программирования
for k in range(1, N_STEPS + 1):
    for x_idx, x in enumerate(x_grid):
        best_cost = float('inf')
        best_y = 0
        
        for y in range(0, x + 1):
            immediate_cost = 3 * (y**2) + 4 * ((x - y)**2)
            
            if k == 1:
                total_cost = immediate_cost
            else:
                next_x = ALPHA * y + BETA * (x - y)
                total_cost = immediate_cost + get_interpolated_cost(next_x, k - 1)
                
            if total_cost < best_cost:
                best_cost = total_cost
                best_y = y
                
        F_table[k, x_idx] = best_cost
        Y_table[k, x_idx] = best_y

# Обратный ход для восстановления оптимальной траектории
trajectory_x = [X_START]
trajectory_y = []
current_x = X_START

for k in range(N_STEPS, 0, -1):
    x_idx_rounded = int(round(current_x))
    opt_y = Y_table[k, x_idx_rounded]
    trajectory_y.append(opt_y)
    
    next_x = ALPHA * opt_y + BETA * (current_x - opt_y)
    trajectory_x.append(next_x)
    current_x = next_x

print("--- ЗАДАЧА 2: РЕЗУЛЬТАТЫ ---")
for step in range(N_STEPS):
    print(f"Шаг {step+1}: Состояние X = {trajectory_x[step]:.2f}, Управление Y = {trajectory_y[step]:.2f}")
print(f"Конечное состояние X: {trajectory_x[-1]:.2f}")
print(f"Глобальный минимум функции F_3(10) = {F_table[N_STEPS, 10]:.2f}")

--- ЗАДАЧА 2: РЕЗУЛЬТАТЫ ---
Шаг 1: Состояние X = 10.00, Управление Y = 6.00
Шаг 2: Состояние X = 4.90, Управление Y = 3.00
Шаг 3: Состояние X = 2.40, Управление Y = 1.00
Шаг 4: Состояние X = 1.22, Управление Y = 1.00
Шаг 5: Состояние X = 0.57, Управление Y = 1.00
Конечное состояние X: 0.21
Глобальный минимум функции F_3(10) = 229.52


Минимум F_3(10): 229.52.
Применение принципа Беллмана к нелинейной задаче на дискретной сетке позволило найти безусловный глобальный минимум целевой функции. Введение механизма линейной интерполяции минимизировало вычислительную ошибку при переходах в нецелочисленные состояния (X = 4.9, X=2.4), обеспечив высокую точность построения оптимальной траектории Y = {6, 3, 1} без необходимости избыточного дробления координатной сетки.

In [8]:
# Исходная матрица: {индекс: (время_А, время_В)}
elements_data = {1: (4, 6), 2: (8, 3), 3: (2, 7), 4: (6, 2), 5: (3, 8), 6: (7, 4), 7: (5, 1), 8: (9, 5)}

def solve_johnson(data):
    left_seq = []
    right_seq = []
    remaining = data.copy()
    
    # Алгоритм сортировки Джонсона
    while remaining:
        min_el = min(remaining, key=lambda k: min(remaining[k]))
        val_A, val_B = remaining[min_el]
        
        if val_A <= val_B:
            left_seq.append(min_el)
        else:
            right_seq.insert(0, min_el)
            
        remaining.pop(min_el)
    return left_seq + right_seq

opt_sequence = solve_johnson(elements_data)

time_A = 0
time_B = 0
idle_B = 0

print("--- ЗАДАЧА 3: РЕЗУЛЬТАТЫ ---")
for el in opt_sequence:
    a_dur, b_dur = elements_data[el]
    start_A = time_A
    end_A = time_A + a_dur
    time_A = end_A
    
    start_B = max(end_A, time_B)
    if start_B > time_B:
        idle_B += (start_B - time_B)
    end_B = start_B + b_dur
    time_B = end_B
    
    print(f"Элемент {el}: Фаза А [{start_A:2d} -> {end_A:2d}], Фаза В [{start_B:2d} -> {end_B:2d}]")

print(f"\nОптимальная последовательность: {opt_sequence}")
print(f"Минимальное суммарное время (Makespan): {time_B}")
print(f"Суммарное время простоя оператора В: {idle_B}")

--- ЗАДАЧА 3: РЕЗУЛЬТАТЫ ---
Элемент 3: Фаза А [ 0 ->  2], Фаза В [ 2 ->  9]
Элемент 5: Фаза А [ 2 ->  5], Фаза В [ 9 -> 17]
Элемент 1: Фаза А [ 5 ->  9], Фаза В [17 -> 23]
Элемент 8: Фаза А [ 9 -> 18], Фаза В [23 -> 28]
Элемент 6: Фаза А [18 -> 25], Фаза В [28 -> 32]
Элемент 2: Фаза А [25 -> 33], Фаза В [33 -> 36]
Элемент 4: Фаза А [33 -> 39], Фаза В [39 -> 41]
Элемент 7: Фаза А [39 -> 44], Фаза В [44 -> 45]

Оптимальная последовательность: [3, 5, 1, 8, 6, 2, 4, 7]
Минимальное суммарное время (Makespan): 45
Суммарное время простоя оператора В: 9


Комбинаторный алгоритм Джонсона математически точно упорядочил элементы, минимизируя время простоя второго оператора. Элементы с минимальными значениями на первой фазе выстроились в начале очереди (левый блок), а с минимальными значениями на второй фазе — в конце (правый блок). Найденная перестановка сокращает суммарный временной интервал до теоретического минимума T = 45.

In [9]:
import numpy as np

N_STEPS = 5
T_MAX = 5
INITIAL_T = 2

def R(t): return 28 - 2.8 * t
def C(t): return 4.5 + 3.2 * t
R_0 = 28

# Таблицы значений
F = np.zeros((N_STEPS + 1, T_MAX + 2))
policy = {}

# Прямой ход Беллмана
for k in range(1, N_STEPS + 1):
    for t in range(1, T_MAX + 1):
        val_keep = R(t) + F[k-1, t+1]
        val_replace = R_0 - C(t) + F[k-1, 1]
        
        if val_keep >= val_replace:
            F[k, t] = val_keep
            policy[(k, t)] = "Сохранить"
        else:
            F[k, t] = val_replace
            policy[(k, t)] = "Обновить"

# Обратный ход восстановления управления
curr_t = INITIAL_T
actions = []

for k in range(N_STEPS, 0, -1):
    action = policy[(k, curr_t)]
    actions.append((curr_t, action))
    curr_t = curr_t + 1 if action == "Сохранить" else 1

print("--- ЗАДАЧА 4: РЕЗУЛЬТАТЫ ---")
for idx, (t_val, act) in enumerate(actions):
    print(f"Шаг {idx+1}: Состояние t = {t_val} -> Действие: {act}")
print(f"Максимальный суммарный эффект: {F[N_STEPS, INITIAL_T]:.2f}")

--- ЗАДАЧА 4: РЕЗУЛЬТАТЫ ---
Шаг 1: Состояние t = 2 -> Действие: Обновить
Шаг 2: Состояние t = 1 -> Действие: Сохранить
Шаг 3: Состояние t = 2 -> Действие: Обновить
Шаг 4: Состояние t = 1 -> Действие: Сохранить
Шаг 5: Состояние t = 2 -> Действие: Сохранить
Максимальный суммарный эффект: 107.00


Модель циклического обновления демонстрирует оптимальный баланс между эксплуатацией текущего состояния и затратами на его принудительный сброс. Принудительное обновление состояния на 1 и 3 шагах позволяет избежать экспоненциального штрафа квадратичной функции, удерживая целевую функцию на максимальном уровне 107.

In [12]:
import numpy as np

# -----------------------------
# Параметры задачи
# -----------------------------
Z = 14          # начальный ресурс
N = 3           # число периодов

alpha = 0.65
beta = 0.35

dX = 2          # шаг сетки состояний
dY = 2          # шаг управления

X_grid = np.arange(0, Z + dX, dX)

# -----------------------------
# Функции дохода
# -----------------------------
def g(y):
    return 5 * y + 0.1 * y**2

def h(x_minus_y):
    return 3 * x_minus_y + 0.18 * x_minus_y**2

# -----------------------------
# Таблицы Беллмана
# F[k,i] = F_k(X_i)
# Y_opt[k,i] = оптимальное управление
# -----------------------------
F = np.zeros((N + 1, len(X_grid)))
Y_opt = np.zeros((N + 1, len(X_grid)))

# -----------------------------
# Прямой ход
# -----------------------------
for k in range(1, N + 1):

    for i, x in enumerate(X_grid):

        best_value = -np.inf
        best_y = 0

        for y in np.arange(0, x + dY, dY):

            immediate_income = g(y) + h(x - y)

            if k == 1:
                total_income = immediate_income
            else:
                next_x = alpha * y + beta * (x - y)

                future_income = np.interp(
                    next_x,
                    X_grid,
                    F[k - 1]
                )

                total_income = immediate_income + future_income

            if total_income > best_value:
                best_value = total_income
                best_y = y

        F[k, i] = best_value
        Y_opt[k, i] = best_y

# -----------------------------
# Вывод таблиц Беллмана
# -----------------------------
print("Таблица значений F_k(X):\n")

for k in range(1, N + 1):
    print(f"F_{k}(X)")
    for x, val in zip(X_grid, F[k]):
        print(f"X={x:2.0f}  F={val:8.3f}")
    print()

# -----------------------------
# Таблица оптимальных управлений
# -----------------------------
print("\nОптимальные управления Y_k(X):\n")

for k in range(1, N + 1):
    print(f"Y_{k}(X)")
    for x, y in zip(X_grid, Y_opt[k]):
        print(f"X={x:2.0f}  Y*={y:2.0f}")
    print()

# -----------------------------
# Обратный ход
# -----------------------------
trajectory = []

current_x = Z

for k in range(N, 0, -1):

    y_star = np.interp(current_x, X_grid, Y_opt[k])

    trajectory.append((k, current_x, y_star))

    current_x = alpha * y_star + beta * (current_x - y_star)

trajectory.reverse()

# -----------------------------
# Вывод траектории
# -----------------------------
print("\nОптимальная траектория:\n")

for k, x, y in trajectory:
    print(
        f"Период {k}: "
        f"X = {x:.3f}, "
        f"Y* = {y:.3f}"
    )

print("\nМаксимальный суммарный доход:")
print(f"F_{N}({Z}) = {F[N, -1]:.3f}")

Таблица значений F_k(X):

F_1(X)
X= 0  F=   0.000
X= 2  F=  10.400
X= 4  F=  21.600
X= 6  F=  33.600
X= 8  F=  46.400
X=10  F=  60.000
X=12  F=  74.400
X=14  F=  89.600

F_2(X)
X= 0  F=   0.000
X= 2  F=  17.160
X= 4  F=  35.360
X= 6  F=  54.640
X= 8  F=  75.200
X=10  F=  96.800
X=12  F= 119.520
X=14  F= 143.480

F_3(X)
X= 0  F=   0.000
X= 2  F=  21.554
X= 4  F=  44.220
X= 6  F=  68.050
X= 8  F=  93.328
X=10  F= 119.780
X=12  F= 147.544
X=14  F= 176.680


Оптимальные управления Y_k(X):

Y_1(X)
X= 0  Y*= 0
X= 2  Y*= 2
X= 4  Y*= 4
X= 6  Y*= 6
X= 8  Y*= 8
X=10  Y*=10
X=12  Y*=12
X=14  Y*=14

Y_2(X)
X= 0  Y*= 0
X= 2  Y*= 2
X= 4  Y*= 4
X= 6  Y*= 6
X= 8  Y*= 8
X=10  Y*=10
X=12  Y*=12
X=14  Y*=14

Y_3(X)
X= 0  Y*= 0
X= 2  Y*= 2
X= 4  Y*= 4
X= 6  Y*= 6
X= 8  Y*= 8
X=10  Y*=10
X=12  Y*=12
X=14  Y*=14


Оптимальная траектория:

Период 1: X = 5.915, Y* = 5.915
Период 2: X = 9.100, Y* = 9.100
Период 3: X = 14.000, Y* = 14.000

Максимальный суммарный доход:
F_3(14) = 176.680


В результате решения задачи методом динамического программирования были определены оптимальные управления и максимальный суммарный доход. Анализ таблиц Беллмана показал, что на каждом этапе наиболее выгодно направлять весь доступный ресурс на получение непосредственного дохода, то есть выбирать управление Y=X. Восстановленная оптимальная траектория подтверждает данную стратегию, а максимальный суммарный доход для начального ресурса Z=14 составил F3 (14) = 176.68